In [1]:
import os
import numpy as np
import pickle
import torch
from argparse import ArgumentParser
from tqdm import tqdm
import glob

from articulate.model import ParametricModel
from articulate import math
from config import paths, datasets


import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from typing import List
import random
import lightning as L
from tqdm import tqdm 

import articulate as art
from config import *
from utils import *
from helpers import *


#libraries


import math
import numpy as np
import torch
torch.set_printoptions(sci_mode=False)
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
from typing import List
import random
import lightning as L
from tqdm import tqdm 

import articulate as art
from config import *
from utils import *
from helpers import *


import os
from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer
from articulate import model as art_model
from utils.model_utils import reduced_pose_to_full
from articulate.math import r6d_to_rotation_matrix
from articulate.evaluator import MeanPerJointErrorEvaluator
from articulate.math import RotationRepresentation



In [2]:
class PoseDataset(Dataset):
    def __init__(self, fold: str='train', evaluate: str=None, finetune: str=None):
        super().__init__()
        self.fold = fold
        self.evaluate = evaluate
        self.finetune = finetune
        self.bodymodel = art.model.ParametricModel(paths.smpl_file)
        self.combos = list(amass.combos.items())
        self.data = self._prepare_dataset()

    def _get_data_files(self, data_folder):
        if self.fold == 'train':
            return self._get_train_files(data_folder)
        elif self.fold == 'test':
            return self._get_test_files()
        else:
            raise ValueError(f"Unknown data fold: {self.fold}.")

    def _get_train_files(self, data_folder):
        if self.finetune:
            return [datasets.finetune_datasets[self.finetune]]
        else:
            return [x.name for x in data_folder.iterdir() if not x.is_dir()]

    def _get_test_files(self):
        return [datasets.test_datasets[self.evaluate]]

    def _prepare_dataset(self):
        data_folder = paths.processed_datasets / ('eval' if (self.finetune or self.evaluate) else '')
        data_files = self._get_data_files(data_folder)

        print(f"\n{'='*60}")
        print(f"Loading datasets for {self.fold.upper()} mode")
        print(f"Datasets: {data_files}")
        print(f"{'='*60}\n")

        data = {key: [] for key in [
            'imu_inputs', 'pose_outputs', 'joint_outputs',
            'tran_outputs', 'vel_outputs', 'foot_outputs'
        ]}

        for data_file in tqdm(data_files):
            try:
                file_data = torch.load(data_folder / data_file)
                self._process_file_data(file_data, data)
            except Exception as e:
                print(f"Error processing {data_file}: {e}")

      
        return data

    def _process_file_data(self, file_data, data):
        accs, oris, poses, trans = file_data['acc'], file_data['ori'], file_data['pose'], file_data['tran']
        joints = file_data.get('joint', [None] * len(poses))
        foots = file_data.get('contact', [None] * len(poses))

        for acc, ori, pose, tran, joint, foot in zip(accs, oris, poses, trans, joints, foots):

            acc, ori = acc[:, :5]/amass.acc_scale, ori[:, :5]

            pose_global, joint = self.bodymodel.forward_kinematics(
                pose=pose.view(-1, 216)
            )

            pose = pose if self.evaluate else pose_global.view(-1, 24, 3, 3)
            joint = joint.view(-1, 24, 3)

            

            self._process_combo_data(acc, ori, pose, joint, tran, foot, data)

    def _process_combo_data(self, acc, ori, pose, joint, tran, foot, data):

        for combo_name, c in self.combos:

            # print("\n" + "="*50)
            # print("Processing combo:", combo_name)

            combo_acc = torch.zeros_like(acc)
            combo_ori = torch.zeros_like(ori)
            combo_acc[:, c] = acc[:, c]
            combo_ori[:, c] = ori[:, c]

            imu_input = torch.cat([combo_acc.flatten(1), combo_ori.flatten(1)], dim=1)

            data_len = len(imu_input) if self.evaluate else datasets.window_length


            for key, value in zip(
                ['imu_inputs', 'pose_outputs', 'joint_outputs', 'tran_outputs'],
                [imu_input, pose, joint, tran]
            ):
                # data[key].extend(torch.split(value, data_len))
                splits = torch.split(value, data_len)

                # remove last if smaller than full window
                if splits[-1].shape[0] < data_len:
                    splits = splits[:-1]

                data[key].extend(splits)

            if not (self.evaluate or self.finetune):
                self._process_translation_data(joint, tran, foot, data_len, data)

    def _process_translation_data(self, joint, tran, foot, data_len, data):

        # print("\nProcessing translation module")

        root_vel = torch.cat((torch.zeros(1, 3), tran[1:] - tran[:-1]))
        vel = torch.cat((torch.zeros(1, 24, 3), torch.diff(joint, dim=0)))
        vel[:, 0] = root_vel

        vel = vel * (datasets.fps / amass.vel_scale)

        vel_splits = torch.split(vel, data_len)

        data['vel_outputs'].extend(vel_splits)
        data['foot_outputs'].extend(torch.split(foot, data_len))

    def __getitem__(self, idx):

        imu = self.data['imu_inputs'][idx].float()
        joint = self.data['joint_outputs'][idx].float()
        tran = self.data['tran_outputs'][idx].float()


        num_pred_joints = len(amass.pred_joints_set)

        pose = art.math.rotation_matrix_to_r6d(
            self.data['pose_outputs'][idx]
        ).reshape(-1, num_pred_joints, 6)[:, amass.pred_joints_set] \
         .reshape(-1, 6*num_pred_joints)


        if self.evaluate or self.finetune:
            return imu, pose, joint, tran

        vel = self.data['vel_outputs'][idx].float()
        contact = self.data['foot_outputs'][idx].float()

        return imu, pose, joint, tran, vel, contact

    def __len__(self):
        return len(self.data['imu_inputs'])
    


def pad_seq(batch):
    """Pad sequences to same length for RNN."""
    def _pad(sequence):
        padded = nn.utils.rnn.pad_sequence(sequence, batch_first=True)
        lengths = [seq.shape[0] for seq in sequence]
        return padded, lengths

    inputs, poses, joints, trans = zip(*[(item[0], item[1], item[2], item[3]) for item in batch])
    inputs, input_lengths = _pad(inputs)
    poses, pose_lengths = _pad(poses)
    joints, joint_lengths = _pad(joints)
    trans, tran_lengths = _pad(trans)
    
    outputs = {'poses': poses, 'joints': joints, 'trans': trans}
    output_lengths = {'poses': pose_lengths, 'joints': joint_lengths, 'trans': tran_lengths}

    if len(batch[0]) > 5: # include velocity and foot contact, if available
        vels, foots = zip(*[(item[4], item[5]) for item in batch])

        # foot contact 
        foot_contacts, foot_contact_lengths = _pad(foots)
        outputs['foot_contacts'], output_lengths['foot_contacts'] = foot_contacts, foot_contact_lengths

        # root velocities
        vels, vel_lengths = _pad(vels)
        outputs['vels'], output_lengths['vels'] = vels, vel_lengths

    return (inputs, input_lengths), (outputs, output_lengths)


class PoseDataModule(L.LightningDataModule):
    def __init__(self, finetune: str = None):
        super().__init__()
        self.finetune = finetune
        self.hypers = finetune_hypers if self.finetune else train_hypers

    def setup(self, stage: str):
        if stage == 'fit':
            dataset = PoseDataset(fold='train', finetune=self.finetune)
            train_size = int(0.9 * len(dataset))
            val_size = len(dataset) - train_size
            self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])
        elif stage == 'test':
            self.test_dataset = PoseDataset(fold='test', finetune=self.finetune)

    def _dataloader(self, dataset):
        return DataLoader(
            dataset, 
            batch_size=self.hypers.batch_size, 
            collate_fn=pad_seq, 
            num_workers=0, #self.hypers.num_workers, 
            shuffle=True, 
            drop_last=True
        )

    def train_dataloader(self):
        return self._dataloader(self.train_dataset)

    def val_dataloader(self):
        return self._dataloader(self.val_dataset)

    def test_dataloader(self):
        return self._dataloader(self.test_dataset)

In [3]:
datamodule = PoseDataModule(finetune=None)
datamodule.setup(stage='fit')

# Get train loader
train_loader = datamodule.train_dataloader()
val_loader = datamodule.val_dataloader()
print("\nTotal training batches:", len(train_loader))
print("Total validation batches:", len(val_loader))


Loading datasets for TRAIN mode
Datasets: ['ACCAD.pt', 'BioMotionLab_NTroje.pt', 'BMLhandball.pt', 'BMLmovi.pt', 'CMU.pt', 'DanceDB.pt', 'DFaust_67.pt', 'dip_test.pt', 'dip_train.pt', 'Eyes_Japan_Dataset.pt', 'HUMAN4D.pt', 'HumanEva.pt', 'MPI_HDM05.pt', 'MPI_Limits.pt', 'MPI_mosh.pt', 'SFU.pt', 'SSM_synced.pt', 'TCD_handMocap.pt', 'TotalCapture.pt', 'Transitions_mocap.pt']



  0%|          | 0/20 [00:00<?, ?it/s]C:\Users\degu03\AppData\Local\Temp\ipykernel_12204\1115591372.py:44: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  file_data = torch.lo


Total training batches: 1171
Total validation batches: 130


In [4]:

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float32)
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Add positional encoding to input.

        x: (B, T, D)
        """
        T = x.size(1)
        return x + self.pe[:T].unsqueeze(0)


class TemporalTransformerDenoiser(nn.Module):
    def __init__(
        self,
        pose_dim: int,
        tran_dim: int = 3,
        d_model: int = 256,
        nhead: int = 8,
        num_layers: int = 4,
        dim_feedforward: int = 512,
        dropout: float = 0.1,
    ):
        """Temporal Transformer denoiser for pose + translation.

        Inputs:  pose (B, T, pose_dim), tran (B, T, tran_dim)
        Outputs: denoised pose (B, T, pose_dim), denoised tran (B, T, tran_dim)
        """
        super().__init__()
        self.pose_dim = pose_dim
        self.tran_dim = tran_dim
        combined_dim = pose_dim + tran_dim

        self.in_proj = nn.Linear(combined_dim, d_model)
        self.pos_enc = SinusoidalPositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # separate output heads for pose and translation
        self.pose_head = nn.Linear(d_model, pose_dim)
        self.tran_head = nn.Linear(d_model, tran_dim)

    def forward(self, pose: torch.Tensor, tran: torch.Tensor):
        # pose: (B, T, pose_dim), tran: (B, T, tran_dim)
        x = torch.cat([pose, tran], dim=-1)  # (B, T, pose_dim + tran_dim)
        h = self.in_proj(x)
        h = self.pos_enc(h)
        h = self.encoder(h)  # (B, T, d_model)
        return self.pose_head(h), self.tran_head(h)  # pose_pred, tran_pred
    

In [5]:
   
def validate(val_loader, model, device, mpj_eval):
    from utils.model_utils import reduced_pose_to_full
    from articulate.math import r6d_to_rotation_matrix
    import torch
    import torch.nn.functional as F

    model.eval()

    val_weighted_loss = 0.0
    val_mpj_sum = 0.0
    val_total_frames = 0

    with torch.no_grad():
        for batch in val_loader:

            (inputs, input_lengths), (outputs, output_lengths) = batch

            x = outputs["poses"].to(device)
            t = outputs["trans"].to(device)
            lengths = torch.as_tensor(output_lengths["poses"], device=device)

            B, T, F = x.shape

            # NO noise during validation
            pose_pred, tran_pred = model(x, t)

            pose_loss = nn.functional.mse_loss(pose_pred, x, reduction="none")
            tran_loss = nn.functional.mse_loss(tran_pred, t, reduction="none")

            mask = torch.arange(T, device=device)[None, :] < lengths[:, None]
            pose_mask = mask.unsqueeze(-1).float()
            tran_mask = mask.unsqueeze(-1).float()

            masked_pose_loss = (pose_loss * pose_mask).sum()
            masked_tran_loss = (tran_loss * tran_mask).sum()
            masked_loss = masked_pose_loss + masked_tran_loss
            num_valid = pose_mask.sum()

            # compute MPJ error for valid frames
            if num_valid > 0:
                if F % 9 == 0:
                    in_rep = 9
                elif F % 6 == 0:
                    in_rep = 6
                else:
                    raise RuntimeError(f"Unknown pose feature dim F={F}")

                num_joints = F // in_rep
                pose_pred_flat = pose_pred.view(B * T, num_joints, in_rep)
                pose_true_flat = x.view(B * T, num_joints, in_rep)
                valid_idx = mask.view(-1).bool()

                if valid_idx.any():
                    if in_rep == 6:
                        pose_pred_mat = r6d_to_rotation_matrix(pose_pred_flat)
                        pose_true_mat = r6d_to_rotation_matrix(pose_true_flat)
                    else:
                        pose_pred_mat = pose_pred_flat.view(-1, num_joints, 3, 3)
                        pose_true_mat = pose_true_flat.view(-1, num_joints, 3, 3)

                    pose_pred_mat = pose_pred_mat[valid_idx]
                    pose_true_mat = pose_true_mat[valid_idx]

                    # model joint count
                    try:
                        j, _ = mpj_eval.model.get_zero_pose_joint_and_vertex(None)
                        model_num_j = j.shape[1] if j.dim() == 3 else j.shape[0]
                    except Exception:
                        model_num_j = pose_pred_mat.shape[1]

                    # expand reduced -> full (fix for shape mismatch)
                    if pose_pred_mat.shape[1] != model_num_j:
                        N_valid = pose_pred_mat.shape[0]
                        pose_pred_full = reduced_pose_to_full(pose_pred_mat.unsqueeze(1)).view(N_valid, model_num_j, 3, 3)
                        pose_true_full = reduced_pose_to_full(pose_true_mat.unsqueeze(1)).view(N_valid, model_num_j, 3, 3)
                        pose_pred_mat = pose_pred_full
                        pose_true_mat = pose_true_full

                    mpj_tensor = mpj_eval(pose_pred_mat, pose_true_mat)
                    val_mpj_sum += mpj_tensor[0].item() * valid_idx.sum().item()

            val_weighted_loss += masked_loss.item()
            val_total_frames += num_valid.item()

    if val_total_frames > 0:
        val_loss = val_weighted_loss / val_total_frames
        val_mpj = val_mpj_sum / val_total_frames
    else:
        val_loss = 0.0
        val_mpj = 0.0

    print(f"Validation Loss: {val_loss:.6f} | MPJPE: {val_mpj:.6f}")

    return val_loss, val_mpj

In [6]:
from utils.model_utils import reduced_pose_to_full
from articulate.math import r6d_to_rotation_matrix

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = TemporalTransformerDenoiser(
        pose_dim=144,
        tran_dim=3,
        d_model=128,
        nhead=4,
        num_layers=2,
        dim_feedforward=256,
        dropout=0.1,
    ).to(device)

model.load_state_dict(torch.load("temporal_transformer_model_best_both.pth", map_location=device))

C:\Users\degu03\AppData\Local\Temp\ipykernel_12204\2270419320.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("temporal_transformer_mod

<All keys matched successfully>

In [8]:
mpj_eval = MeanPerJointErrorEvaluator(
official_model_file=paths.smpl_file,
rep=RotationRepresentation.ROTATION_MATRIX,
device=device
    )

val_loss, val_mpj = validate(
    val_loader=val_loader,
    model=model,
    device=device,
    mpj_eval=mpj_eval
)
print(f"Validation Loss: {val_loss:.6f} | MPJPE: {val_mpj:.6f}")

Validation Loss: 0.038448 | MPJPE: 0.017195
Validation Loss: 0.038448 | MPJPE: 0.017195


In [10]:
from articulate.math import r6d_to_rotation_matrix
from viewers.smpl_viewer import SMPLViewer
import os
os.environ["GT"] = "1"
model.eval()

# 180° rotation around X-axis to flip the body upright
flip_rot = torch.eye(3, device=device)
flip_rot[1, 1] = -1
flip_rot[2, 2] = -1

k = 0

for i, batch in enumerate(val_loader):
    if i == k:
        break

(inputs, input_lengths), (outputs, output_lengths) = batch

x = outputs["poses"].to(device)
trans = outputs["trans"].to(device)

B, T, Fdim = x.shape

# noise = torch.randn_like(x) * 0.01
# noisy_x = x + noise


pose_noise = torch.randn_like(x) * 0.01
tran_noise = torch.randn_like(trans) * 0.01
noisy_x = x + pose_noise
noisy_t = trans + tran_noise

with torch.no_grad():
    pose_pred, tran_pred = model(noisy_x, noisy_t)

gt_batch = x
pred_batch = pose_pred
pred_batch_tran = tran_pred
tran_batch = trans
lengths = output_lengths["poses"]

viewer = SMPLViewer(fps=25)
bodymodel = viewer.bodymodel

batch_size = 2

for b in range(batch_size):
    L = int(lengths[b])

    gt_pose_r6d   = gt_batch[b, :L]
    pred_pose_r6d = pred_batch[b, :L]
    tran_gt       = tran_batch[b, :L]
    tran_pred     = pred_batch_tran[b, :L]

    # Convert R6D -> global rotation matrices
    gt_pose_rot = r6d_to_rotation_matrix(
        gt_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    pred_pose_rot = r6d_to_rotation_matrix(
        pred_pose_r6d.view(-1, 24, 6)
    ).view(-1, 24, 3, 3)

    # Convert global -> local (viewer applies FK internally)
    gt_pose_local = bodymodel.inverse_kinematics_R(gt_pose_rot)
    pred_pose_local = bodymodel.inverse_kinematics_R(pred_pose_rot)

    # Flip root joint to stand upright
    gt_pose_local[:, 0] = flip_rot @ gt_pose_local[:, 0]
    pred_pose_local[:, 0] = flip_rot @ pred_pose_local[:, 0]

    viewer.view(pred_pose_local, tran_pred, gt_pose_local, tran_gt, with_tran=True)


100%|##########| 125/125 [00:02<00:00, 59.23it/s]
